# Demonstration 

## Setup

First, it is necessary to import packages. `Munch` is a `Dict` that acts like a structure, so we can use dot notation. We'll use this to demonstrate different functions but keep the results organized

In [1]:
import numpy as np
import astropy.units as u
from munch import Munch

import tellurion as tell
import tellurion.ork as tork

Python version: 3.12.8 | packaged by conda-forge | (main, Dec  5 2024, 14:24:40) [GCC 13.3.0]
Java version: 1.8.0_412
Numpy version: 2.2.1
Astropy version: 6.1.7
Orekit version: 12.2


Next we define a specific epoch time `newyear` and a sequence of time steps every 5 minutes for one hour `prop5m1h`

In [2]:
newyear = tell.dttm('2025-01-01T00:00:00')
prop5m1h = np.linspace(5.0*u.minute, 60.0*u.minute, 12) # Step every 5 minutes for an hour

## State and elements


Create the tree structures to hold values

In [3]:
demoa = Munch()
demoa.init = Munch()
demoa.init.check = Munch()

Create a position, velocity and time, representing the __[orbital state vector](https://en.wikipedia.org/wiki/Orbital_state_vectors)__

In [4]:
demoa.init.pos = [5740.13268349, 3314.06715   ,    0.]
demoa.init.vel = [-2.75082684,  4.76457184,  5.50165367]
demoa.init.pvt = tell.pvt((demoa.init.pos, demoa.init.vel, newyear))  # A tuple (Quantity, Time)
demoa.init.pvt

(<Quantity ([5740.13268349, 3314.06715   ,    0.        ], [-2.75082684,  4.76457184,  5.50165367]) (km, km / s)>,
 <Time object: scale='utc' format='datetime64' value=2025-01-01T00:00:00.000000000>)

Transform this into a Kepler element set and display it as a dictionary of dimensioned quantities

In [5]:
demoa.init.kep = tork.kepler(demoa.init.pvt)  # Convert PVT to Kepler elements
tell.splitsq(demoa.init.kep[0])  # Easier to read Kepler elements

{'inc': <Quantity 44.99999999 deg>,
 'argper': <Quantity 5.42238634e-06 deg>,
 'raan': <Quantity 30. deg>,
 'ecc': <Quantity 0.00663034>,
 'sma': <Quantity 6672.37441089 km>,
 'ma': <Quantity -5.3508378e-06 deg>}

## Two-body propagation

Create dictionaries

In [6]:
demoa.prop = Munch()
demoa.prop.check = Munch()

Make the propagation generator, which is necessary to create an ephemeris. A maximum time must be specified; we will make it 1 day.

In [7]:
demoa.prop.gen = tork.generate(demoa.init.pvt, 1*u.day)

Propagate for 1 hour in 5 minute steps, and include the initial state in the ephemeris table; note the middle (`Y`) value for the position and velocity vectors has been elided, but it is present 

In [8]:
demoa.prop.ephem = tork.propagate(demoa.prop.gen, prop5m1h, True)
demoa.prop.ephem

time,position,velocity
,km,km / s
Time,float64[3],float64[3]
2025-01-01 00:00:00,5740.133 .. 0.000,-2.750827 .. 5.501654
2025-01-01 00:05:00,4581.815 .. 1616.826,-4.891449 .. 5.166423
2025-01-01 00:10:00,2865.500 .. 3036.847,-6.432387 .. 4.203822
2025-01-01 00:15:00,801.265 .. 4088.441,-7.187958 .. 2.736514
2025-01-01 00:20:00,-1359.960 .. 4646.558,-7.074062 .. 0.948420
2025-01-01 00:25:00,-3358.332 .. 4647.312,-6.115174 .. -0.941270
2025-01-01 00:30:00,-4956.792 .. 4094.286,-4.436284 .. -2.706752
2025-01-01 00:35:00,-5968.504 .. 3056.386,-2.243218 .. -4.142426


Altitudes of perigee and apogee; because a two-body propagator is used, they only change due to the slow rotation of the coordinate system

In [9]:
demoa.prop.altperapo = tork.tselements(demoa.prop.ephem, ["altper","altapo"])
demoa.prop.altperapo

time,altper,altapo
,km,km
Time,float64,float64
2025-01-01 00:00:00,249.99783999568038,338.478061778428
2025-01-01 00:05:00,249.99635500008242,338.47545385003275
2025-01-01 00:10:00,249.99821081749442,338.47896186375806
2025-01-01 00:15:00,249.99490359744152,338.4760766057121
2025-01-01 00:20:00,249.99724508334603,338.47847316549996
2025-01-01 00:25:00,249.9949079351658,338.47798071310393
2025-01-01 00:30:00,249.99678238372505,338.4786557904799
2025-01-01 00:35:00,249.99724316924065,338.47926235181654


## Data at single points

Propagate to a single time with the same generator; propagate to 12 hours, which is within the maximum allowable time of 1 day for this generator

In [10]:
tork.propagate(demoa.prop.gen, 12*u.hour)

(<Quantity ([ 6121.0529455 ,  2317.14340635, -1053.82141834], [-1.17605108,  5.51222363,  5.36175123]) (km, km / s)>,
 <Time object: scale='utc' format='datetime' value=2025-01-01 12:00:00>)

Find the initial state directly from the generator as Cartesian PVT and Kepler elements

In [11]:
demoa.prop.gen.pvt()

(<Quantity ([5740.13268349, 3314.06715   ,    0.        ], [-2.75082684,  4.76457184,  5.50165367]) (km, km / s)>,
 <Time object: scale='utc' format='datetime' value=2025-01-01 00:00:00>)

In [12]:
demoa.prop.gen.kepler()

(<Quantity (44.99999999, 5.42238634e-06, 30., 0.00663034, 6672.37441089, -5.3508378e-06) (deg, deg, deg, , km, deg)>,
 <Time object: scale='utc' format='datetime' value=2025-01-01 00:00:00>)

In [13]:
tell.splitsq(demoa.prop.gen.kepler()[0]) # Easier to read form for orbital elements

{'inc': <Quantity 44.99999999 deg>,
 'argper': <Quantity 5.42238634e-06 deg>,
 'raan': <Quantity 30. deg>,
 'ecc': <Quantity 0.00663034>,
 'sma': <Quantity 6672.37441089 km>,
 'ma': <Quantity -5.3508378e-06 deg>}

Get a single point from the ephemeris table in multiple ways; unfortunately relative time doesn't work

In [14]:
tell.pvt(demoa.prop.ephem, 5)

(<Quantity ([-3358.33216616,  3427.3199443 ,  4647.31222039], [-6.11517395, -4.61748178, -0.94126955]) (km, km / s)>,
 <Time object: scale='utc' format='datetime' value=2025-01-01 00:25:00>)

In [15]:
tell.pvt(demoa.prop.ephem,'2025-01-01 00:25:00')

(<Quantity ([-3358.33216616,  3427.3199443 ,  4647.31222039], [-6.11517395, -4.61748178, -0.94126955]) (km, km / s)>,
 <Time object: scale='utc' format='datetime' value=2025-01-01 00:25:00>)

In [16]:
# posvel.pvt(demoa.prop.ephem, 25*u.min) # Does not work, but should